# Senthium AI - Exploratory Data Analysis

This notebook demonstrates how to analyze telemetry data collected by the DataLogger.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import Senthium AI modules
import sys
sys.path.insert(0, '../')
from senthium_ai.utils import FeatureExtractor

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

Load telemetry data from CSV files in the data/raw directory.

In [ ]:
# Example: Load a single CSV file
data_dir = Path('../data/raw')
csv_files = list(data_dir.glob('*.csv'))

if csv_files:
    df = pd.read_csv(csv_files[0])
    print(f"Loaded {len(df)} samples from {csv_files[0].name}")
    print(f"\nColumns: {list(df.columns)}")
    df.head()
else:
    print("No CSV files found in data/raw. Run the data logger first!")
    df = None

## Basic Statistics

In [ ]:
if df is not None:
    print("Data Summary:")
    print(df.describe())
    
    print("\nTask Label Distribution:")
    print(df['task_label'].value_counts())

## Visualizations

### CPU and Memory Usage Over Time

In [ ]:
if df is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # CPU usage
    axes[0].plot(df['cpu_mean'], label='CPU Mean', alpha=0.7)
    axes[0].fill_between(range(len(df)), 
                         df['cpu_mean'] - df['cpu_std'], 
                         df['cpu_mean'] + df['cpu_std'], 
                         alpha=0.3, label='±1 std')
    axes[0].set_ylabel('CPU Usage (%)')
    axes[0].set_title('CPU Usage Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Memory usage
    axes[1].plot(df['mem_percent'], label='Memory', color='orange', alpha=0.7)
    axes[1].set_ylabel('Memory Usage (%)')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_title('Memory Usage Over Time')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Network and Disk I/O

In [ ]:
if df is not None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Network I/O
    axes[0].plot(df['net_rx_rate'] / 1024, label='RX (KB/s)', alpha=0.7)
    axes[0].plot(df['net_tx_rate'] / 1024, label='TX (KB/s)', alpha=0.7)
    axes[0].set_ylabel('Network I/O (KB/s)')
    axes[0].set_title('Network I/O Over Time')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Disk I/O
    axes[1].plot(df['disk_read_rate'] / 1024, label='Read (KB/s)', alpha=0.7)
    axes[1].plot(df['disk_write_rate'] / 1024, label='Write (KB/s)', alpha=0.7)
    axes[1].set_ylabel('Disk I/O (KB/s)')
    axes[1].set_xlabel('Sample Index')
    axes[1].set_title('Disk I/O Over Time')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### Task Label Analysis

In [ ]:
if df is not None and 'task_label' in df.columns:
    # Group by task label and compute statistics
    task_stats = df.groupby('task_label').agg({
        'cpu_mean': ['mean', 'std'],
        'mem_percent': ['mean', 'std'],
        'net_rx_rate': 'mean',
        'disk_write_rate': 'mean'
    })
    
    print("Statistics by Task Label:")
    print(task_stats)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # CPU by task
    df.boxplot(column='cpu_mean', by='task_label', ax=axes[0])
    axes[0].set_title('CPU Usage by Task Type')
    axes[0].set_xlabel('Task Label')
    axes[0].set_ylabel('CPU Mean (%)')
    
    # Memory by task
    df.boxplot(column='mem_percent', by='task_label', ax=axes[1])
    axes[1].set_title('Memory Usage by Task Type')
    axes[1].set_xlabel('Task Label')
    axes[1].set_ylabel('Memory (%)')
    
    plt.tight_layout()
    plt.show()

## Feature Extraction

Extract ML features using the FeatureExtractor.

In [ ]:
if csv_files:
    extractor = FeatureExtractor(window_size=6, stride=1)
    X, y_class, y_time = extractor.process_file(str(csv_files[0]))
    
    print(f"Extracted {len(X)} windows")
    print(f"Feature shape: {X.shape}")
    print(f"Class distribution: {np.bincount(y_class)}")
    
    # Visualize feature distributions
    feature_names = extractor.get_feature_names()
    
    # Plot first 10 features
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for i in range(min(10, len(feature_names))):
        axes[i].hist(X[:, i], bins=30, alpha=0.7)
        axes[i].set_title(feature_names[i], fontsize=10)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Feature Correlation

In [ ]:
if csv_files and len(X) > 0:
    # Create correlation matrix
    feature_df = pd.DataFrame(X, columns=feature_names)
    corr_matrix = feature_df.corr()
    
    # Plot heatmap
    plt.figure(figsize=(16, 14))
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
                square=True, linewidths=0.5)
    plt.title('Feature Correlation Matrix', fontsize=16)
    plt.tight_layout()
    plt.show()

## Conclusion

This notebook provides basic exploratory analysis of the telemetry data. Use these insights to:
- Identify patterns in different task types
- Optimize feature engineering
- Set appropriate thresholds for fuzzy logic
- Validate data quality before training

Next steps:
1. Collect more diverse data samples
2. Train the ANN model using `train_model.py`
3. Evaluate model performance
4. Deploy the trained model with Senthium AI